In [124]:
# Third-party core
import boto3
import pandas as pd
from sqlalchemy import create_engine


# SageMaker
import sagemaker
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.processing import ScriptProcessor
from sagemaker.pytorch.estimator import PyTorch
from sagemaker.inputs import TrainingInput

In [125]:
%store -r
%store

Stored variables and their in-db values:
bucket                                     -> 'sagemaker-us-east-1-298748835671'
database_name                              -> 'cat_landmarking'
ingest_create_athena_db_passed             -> True
ingestion_completed                        -> True
landmarks_table                            -> 'cat_annotations'
manifest_table                             -> 'image_manifest'
project_prefix                             -> 'cat-landmarks-project'
s3_athena_results_dir                      -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_processed_cats_prefix                   -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_processed_combined_prefix               -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_raw_cats_prefix                         -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_raw_noncats_prefix                      -> 's3://sagemaker-us-east-1-298748835671/cat-landmar
s3_staging_dir                

In [126]:
bucket = bucket
database_name = database_name
project_prefix = project_prefix
landmarks_table = landmarks_table
s3_staging_dir = s3_staging_dir
s3_raw_cats_prefix = s3_raw_cats_prefix
manifest_table = manifest_table

In [127]:
s3 = boto3.client("s3")
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session._region_name

boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(service_name="sagemaker", 
                                       region_name=region)


# SQL connection specification
engine = create_engine(
    f"awsathena+rest://@athena.{region}.amazonaws.com:443/"
    f"{database_name}"
    f"?s3_staging_dir={s3_staging_dir}"
)

s3_athena_results_dir = f"s3://{bucket}/{project_prefix}/athena/results/"

In [128]:
# Retrieve feature group metadata
feature_groups = sagemaker_client.list_feature_groups()
feature_groups

{'FeatureGroupSummaries': [{'FeatureGroupName': 'landmarks-feature-group-17-18-53-51',
   'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:298748835671:feature-group/landmarks-feature-group-17-18-53-51',
   'CreationTime': datetime.datetime(2026, 2, 17, 18, 53, 51, 418000, tzinfo=tzlocal()),
   'FeatureGroupStatus': 'Created',
   'OfflineStoreStatus': {'Status': 'Active'}}],
 'ResponseMetadata': {'RequestId': '3eb1e991-2718-417e-ba76-26928bfc06df',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '3eb1e991-2718-417e-ba76-26928bfc06df',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '301',
   'date': 'Thu, 19 Feb 2026 01:42:36 GMT'},
  'RetryAttempts': 0}}

In [129]:
# Instantiate Landmarks Feature Group object
fg_name = feature_groups["FeatureGroupSummaries"][0]["FeatureGroupName"]

fg = FeatureGroup(
	name=fg_name,
	sagemaker_session=sagemaker_session
)

fg_refs = fg.describe()["OfflineStoreConfig"]["DataCatalogConfig"]
fg_refs

{'TableName': 'landmarks_feature_group_17_18_53_51_1771354431',
 'Catalog': 'AwsDataCatalog',
 'Database': 'sagemaker_featurestore'}

In [130]:
fg_training_data_query = f"""
	select 
 		remote_path,
   		split,
		width,
		height,
	 	left_eye_x,
	 	left_eye_y,
	    right_eye_x,
	    right_eye_y,
	    mouth_x,
	    mouth_y,
	    left_ear_1_x,
     	left_ear_1_y,
	    left_ear_2_x,
	    left_ear_2_y,
	    left_ear_3_x,
     	left_ear_3_y,
	    right_ear_1_x,
	    right_ear_1_y,
	    right_ear_2_x,
     	right_ear_2_y,
	    right_ear_3_x,
	    right_ear_3_y
 	from {fg_refs["Database"]}.{fg_refs["TableName"]}
"""

landmarks_sample = pd.read_sql(fg_training_data_query + "limit 5", engine)
landmarks_sample

,remote_path,split,width,height,left_eye_x,left_eye_y,right_eye_x,right_eye_y,mouth_x,mouth_y,...,left_ear_2_x,left_ear_2_y,left_ear_3_x,left_ear_3_y,right_ear_1_x,right_ear_1_y,right_ear_2_x,right_ear_2_y,right_ear_3_x,right_ear_3_y
0,s3://sagemaker-us-east-1-298748835671/cat-land...,production,500,494,111,293,154,299,128,336,...,81,217,113,239,165,245,205,227,189,279
1,s3://sagemaker-us-east-1-298748835671/cat-land...,test,500,333,278,121,347,121,317,179,...,220,12,282,61,339,60,397,16,387,76
2,s3://sagemaker-us-east-1-298748835671/cat-land...,production,1024,873,396,321,575,291,509,427,...,216,98,368,204,546,162,638,8,664,213
3,s3://sagemaker-us-east-1-298748835671/cat-land...,train,768,1024,557,351,286,345,448,513,...,680,43,521,158,249,152,59,43,95,303
4,s3://sagemaker-us-east-1-298748835671/cat-land...,production,500,334,145,175,249,169,205,252,...,48,29,133,92,251,85,311,3,302,130


In [131]:
output_s3_uri = f"s3://{bucket}/{project_prefix}/training-data/"

fg_query = fg.athena_query()
fg_query.run(
    query_string=fg_training_data_query,
    output_location=output_s3_uri
)
fg_query.wait()

INFO:sagemaker:Query 8dc31e96-135d-4fe6-b685-85cdaef776af is being executed.
INFO:sagemaker:Query 8dc31e96-135d-4fe6-b685-85cdaef776af successfully executed.


In [132]:
instance_type = "ml.g5.4xlarge"
pipeline_session = PipelineSession()

image_uri = sagemaker.image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.1.0",
    py_version="py310",
    instance_type=instance_type,
    image_scope="training",
)

def make_processor():
    return ScriptProcessor(
        command=["python3"],
        image_uri=image_uri,
        role=role,
        instance_type=instance_type,
        instance_count=1,
        sagemaker_session=pipeline_session,
    )

# Step process for training
step_process_train = ProcessingStep(
    name="KeypointPreprocessingTrain",
    processor=make_processor(),
    inputs=[
        sagemaker.processing.ProcessingInput(
            source=output_s3_uri,          
            destination="/opt/ml/processing/metadata"
        )
    ],
    outputs=[
        sagemaker.processing.ProcessingOutput(
            output_name="train",      
            source="/opt/ml/processing/output"
        )
    ],
    code="./src/keypoint_regression/preprocess.py",
    job_arguments=[
        "--metadata", "/opt/ml/processing/metadata",
        "--output",   "/opt/ml/processing/output",
        "--split",    "train",     
    ],
)

# Step process for validation
step_process_val = ProcessingStep(
    name="KeypointPreprocessingValidation",
    processor=make_processor(),
    inputs=[
        sagemaker.processing.ProcessingInput(
            source=output_s3_uri,
            destination="/opt/ml/processing/metadata"
        )
    ],
    outputs=[
        sagemaker.processing.ProcessingOutput(
            output_name="validation",    
            source="/opt/ml/processing/output"
        )
    ],
    code="./src/keypoint_regression/preprocess.py",
    job_arguments=[
        "--metadata", "/opt/ml/processing/metadata",
        "--output",   "/opt/ml/processing/output",
        "--split",    "validation",
    ],
)

In [133]:
model_path = f"s3://{bucket}/{project_prefix}/models/model_artifacts"

pytorch_train = PyTorch(
    entry_point="train.py",
    source_dir="./src/keypoint_regression",
    role=role,
    framework_version="2.1.0",
    py_version="py310",
    instance_type=instance_type,
    instance_count=1,
    output_path=model_path,
    sagemaker_session=pipeline_session,
    hyperparameters={
        "epochs": 10,
        "learning-rate": 0.001,
        "batch-size": 64,
    },
)

train_args = pytorch_train.fit(
    inputs={
        "train": TrainingInput(
            s3_data=step_process_train.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="application/x-parquet",
        ),
        "validation": TrainingInput(
            s3_data=step_process_val.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="application/x-parquet",
        ),
    }
)

step_train = TrainingStep(
    name="TrainKeypointModel",
    step_args=train_args,
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [ ]:
step_process_val.add_depends_on([step_process_train])

pipeline = Pipeline(
    name="KeypointPipeline",
    steps=[step_process_train, step_process_val, step_train],
    sagemaker_session=pipeline_session,
)

pipeline.upsert(role_arn=role)
execution = pipeline.start()
execution.wait()
print(execution.describe())

In [141]:
for step in execution.list_steps():
    print(f"Step: {step['StepName']}")
    print(f"Status: {step['StepStatus']}")
    if step['StepStatus'] == 'Failed':
        print(f"Failure Reason: {step['FailureReason']}")
    print("---")

Step: TrainKeypointModel
Status: Executing
---
Step: KeypointPreprocessingValidation
Status: Succeeded
---
Step: KeypointPreprocessingTrain
Status: Succeeded
---


In [142]:
# Get the exact S3 path of the parquet file Athena wrote
for step in execution.list_steps():
    if step["StepStatus"] == "Failed":
        print(step)